## Purpose of This Notebook

This notebook serves as an exploratory tool for examining the log files produced during the horse behavioural experiments conducted in October and November 2023. 

It facilitates loading and querying the data that was pushed into the database in `logfile-to-database.ipynb`

In [ ]:
# | echo: false

from pathlib import Path
from pprint import pprint

import duckdb
import pandas as pd
from IPython.display import Markdown, display
from loguru import logger

from horse_logic.utils import set_custom_logger_format

In [ ]:
# | echo: false

import itables.options as opt
from itables import init_notebook_mode

init_notebook_mode(all_interactive=True)  # Display dataframes in a more friendly paginated manner
opt.pageLength = 20


In [ ]:
# | echo: false

set_custom_logger_format()

### Directory information: log file data and outputs

In [ ]:
# | echo: false

DATA_DIR = Path("../data")
EXPERIMENT_TYPE = "RPE"    # or CB

assert DATA_DIR.exists()

DATA_DB  = DATA_DIR / f"Experiments_{EXPERIMENT_TYPE}_2023-Q4.ddb"  # DuckDB database name
db_exists = DATA_DB.exists()

logger.info(f"Database file: {DATA_DB.resolve()}")

In [ ]:
# | echo: false

OUTPUT_DIR = DATA_DIR / f"results/{EXPERIMENT_TYPE}"

assert OUTPUT_DIR.exists()
logger.info(f"Outputs dir: {OUTPUT_DIR.resolve()}")

#### Connect to database

In [ ]:
# | echo: false

def connect_ddb(database=DATA_DB):
    try:
        if db_exists:  
            con = duckdb.connect(database=str(database))
        else:
            logger.error(f"Database file: {database.resolve()} not found.")
    except Exception as e:
        print(f"Error with Database file: {database.resolve()} - Is database already open - check for .wal lock file?")
        logger.error(f"{e}")
    logger.info(f"CONNECTED - Database file: {database.resolve()}")
    return con


con = connect_ddb()


In [ ]:
# Cross-check queries

con.sql("SHOW TABLES;")

In [ ]:
event_type_ddb = con.sql("SELECT DISTINCT EventType FROM Events ORDER BY EventType").df()

In [ ]:
events_ddb = con.sql("SELECT * FROM Events").df()

In [ ]:
event_type_ddb["EventType"].unique()

#### Reconstruct an experiment example

In [ ]:
experiments_df = con.sql("SELECT * FROM Experiments").df()

In [ ]:
experiments_df

In [ ]:
con.sql(
    """
    SELECT
        ExperimentID, 
        SubjectName, 
        SubjectNumber, 
        SessionNumber,
        ExperimentType, 
        Comment, 
        DateTime, 
        LogFileName
    FROM Experiments 
    WHERE ExperimentID = 128;
"""
).df()

In [ ]:
con.sql("""
    SELECT 
        -- Trials.TrialID,
        Trials.TrialNumber,
        -- TrialStartTime,
        -- TrialEndTime,
        -- EventID, 
        EventNumber, 
        EventTime, 
        EventType,
        EventElapsedTime
    FROM Experiments
    INNER JOIN Trials ON Experiments.ExperimentID = Trials.ExperimentID
    INNER JOIN Events ON Trials.TrialID = Events.TrialID
    WHERE Experiments.ExperimentID = 128;
""").df()

In [ ]:
con.sql("""
    SELECT 
        -- Experiments.SubjectName,
        -- Experiments.SessionNumber,
        -- Trials.TrialID,
        Trials.TrialNumber,
        -- TrialStartTime,
        -- TrialEndTime,
        -- EventID, 
        EventNumber, 
        EventTime, 
        EventType,
        EventElapsedTime
    FROM Experiments
    INNER JOIN Trials ON Experiments.ExperimentID = Trials.ExperimentID
    INNER JOIN Events ON Trials.TrialID = Events.TrialID
    WHERE Experiments.SubjectName = 'gio' AND  Experiments.SessionNumber = 2;
""").df()

In [ ]:
con.close()